# 04 — Live portfolio report

This notebook shows the current live-report workflow used by both the CLI and PyQt dashboard. Cobasket reads persistent portfolio state, re-evaluates the watchlist using current cached/downloaded prices, applies the saved probability calibration when available, and produces a serializable report.

It does not place brokerage orders.


In [ ]:
from pathlib import Path

from cobasket.workflow import PortfolioAnalyzer, PortfolioConfig


## 1. Load an existing portfolio configuration

Using the same `portfolio.json` as the CLI and GUI avoids maintaining several copies of your state. Relative watchlist and calibration paths are resolved relative to the portfolio file in the user-facing workflows.


In [ ]:
portfolio_path = Path("../portfolio.json").resolve()
config = PortfolioConfig.load(portfolio_path)
config


`holdings` records current share quantities. The watchlist is independent of holdings, so a ticker with quantity zero can remain under observation for a possible later re-entry.


## 2. Generate the report

For normal use you can run:

```bash
cobasket-report --portfolio portfolio.json --output report.json
```

The equivalent Python workflow is:


In [ ]:
report = PortfolioAnalyzer().run(config)
report.table()


In [ ]:
report.warnings


When `calibration_path` is set, the `Probability` column is an empirical basket-relative outperformance probability with uncertainty and sample-count diagnostics. If no calibration is supplied, Cobasket falls back to raw evidence thresholds and reports that explicitly as a warning.


## 3. Inspect basket diagnostics and report metadata


In [ ]:
report.basket_diagnostics


In [ ]:
report.metadata


## 4. Save the report for the GUI

The GUI has separate inputs for `portfolio.json` and a saved report. `portfolio.json` is used for analysis/editing; `report.json` is a displayable snapshot.


In [ ]:
report.save("../report.json")
report.to_dict()


## 5. Recommended live workflow

1. Keep `portfolio.json` and the watchlist up to date.
2. Refresh the probability calibration when the monitored basket set or modelling assumptions change materially.
3. Run a fresh analysis every few days or whenever you want to review the portfolio.
4. Inspect the probability, uncertainty, explanation, basket membership, and warnings rather than acting on the recommendation label alone.
5. Record actual decisions in Recommendation History if you want to compare model suggestions with what you did.

Notebook 10 is the diagnostic check on whether the current calibration is credible enough to use.
